# Gold Layer - Medication Price by State

## Description
This notebook creates a Gold-level aggregated dataset showing medication price statistics per U.S. state.

The dataset aggregates pharmacy inventory price data to provide a state-level view of medication pricing across pharmacies.

## Source Tables
- capstone.silver.inventory  
- capstone.silver.medications  
- capstone.silver.pharmacy  

## Target Table
- capstone.gold.medication_price_state_gold

## Transformations
- Join inventory with pharmacy to obtain state  
- Join medications to retrieve medication name  
- Filter invalid or missing price values  
- Aggregate by state and medication  
- Calculate average medication price  
- Calculate minimum medication price  
- Calculate maximum medication price

## 1. Setup Environment

In [0]:
# Setup environment
# Use capstone catalog and ensure Gold schema exists

spark.sql("USE CATALOG capstone")
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

## 2. Create Gold Table

In [0]:
%sql
CREATE OR REPLACE TABLE capstone.gold.medication_price_state_gold
USING DELTA
AS

SELECT
    state,
    medication_id,
    medication_name,

    AVG(price) AS avg_price,
    MIN(price) AS min_price,
    MAX(price) AS max_price

FROM capstone.gold.disease_medication_pharmacy_gold

WHERE
    price IS NOT NULL
    AND price >= 0

GROUP BY
    state,
    medication_id,
    medication_name;

In [0]:
%sql
-- Preview the Gold dataset

SELECT *
FROM capstone.gold.medication_price_state_gold
ORDER BY avg_price DESC


## 3. Document Table **Columns**

In [0]:
%sql

-- ============================================================
-- 3. Document Table Columns
-- Add descriptions using ALTER for better data catalog clarity
-- ============================================================

ALTER TABLE capstone.gold.medication_price_state_gold
SET TBLPROPERTIES (
  'comment' = 'Gold dataset providing medication price statistics (average, minimum, maximum) aggregated by U.S. state'
);

ALTER TABLE capstone.gold.medication_price_state_gold
ALTER COLUMN state COMMENT 'U.S. state where the pharmacy selling the medication is located';

ALTER TABLE capstone.gold.medication_price_state_gold
ALTER COLUMN medication_id COMMENT 'Unique identifier for the medication';

ALTER TABLE capstone.gold.medication_price_state_gold
ALTER COLUMN medication_name COMMENT 'Medication name used for reporting and analytical filtering';

ALTER TABLE capstone.gold.medication_price_state_gold
ALTER COLUMN avg_price COMMENT 'Average medication price across pharmacies within the state';

ALTER TABLE capstone.gold.medication_price_state_gold
ALTER COLUMN min_price COMMENT 'Minimum medication price observed in the state';

ALTER TABLE capstone.gold.medication_price_state_gold
ALTER COLUMN max_price COMMENT 'Maximum medication price observed in the state';

## 4. Gold Data Quality Checks

In [0]:
%sql

SELECT
'Total Rows' AS check_type,
COUNT(*) AS result
FROM capstone.gold.medication_price_state_gold

UNION ALL

SELECT
'Missing State',
COUNT(*)
FROM capstone.gold.medication_price_state_gold
WHERE state IS NULL

UNION ALL

SELECT
'Missing Medication ID',
COUNT(*)
FROM capstone.gold.medication_price_state_gold
WHERE medication_id IS NULL

UNION ALL

SELECT
'Negative Prices',
COUNT(*)
FROM capstone.gold.medication_price_state_gold
WHERE avg_price < 0

UNION ALL

SELECT
'Duplicate State + Medication',
COUNT(*)
FROM (
    SELECT state, medication_id
    FROM capstone.gold.medication_price_state_gold
    GROUP BY state, medication_id
    HAVING COUNT(*) > 1
);

## 5. Validate Gold Table

In [0]:
%sql

SELECT
COUNT(*) AS total_rows,
COUNT(DISTINCT state) AS distinct_states,
COUNT(DISTINCT medication_id) AS distinct_medications
FROM capstone.gold.medication_price_state_gold;

## 6. KPI - Medication Price Comparison by State and Medication
Price comparison across States
Filtering by medication

In [0]:
%sql
-- KPI: Medication Price Comparison by State
-- Purpose:
-- Allow analysts to compare medication prices across states
-- for each medication

CREATE OR REPLACE VIEW capstone.kpi.kpi_medication_price_comparison_by_state AS

SELECT
    state,
    medication_name,
    avg_price,
    min_price,
    max_price
FROM capstone.gold.medication_price_state_gold
ORDER BY state, medication_name;
SELECT *
FROM capstone.kpi.kpi_medication_price_comparison_by_state;

## 7. KPI - Average Medication Price by State


In [0]:
%sql
-- KPI: Average Medication Price by State
-- Purpose:
-- Support the business requirement "Average medication price by State"
-- Enables price comparison across U.S. states

CREATE OR REPLACE VIEW capstone.kpi.kpi_avg_medication_price_by_state AS

SELECT
    state,
    AVG(avg_price) AS average_medication_price
FROM capstone.gold.medication_price_state_gold
GROUP BY state
ORDER BY average_medication_price DESC;
SELECT *
FROM capstone.kpi.kpi_avg_medication_price_by_state;